# **Documentación del Sistema de Reservas de Hotel**

Este documento proporciona una guía completa para la creación y uso de la base de datos **ReservasHotel**, diseñada para gestionar reservas de habitaciones, clientes, servicios y más en un hotel. A continuación, se detallan las tablas, procedimientos almacenados y triggers que componen el sistema.

---

## **Tablas de la Base de Datos**

### **1. Tabla `Habitaciones`**
Almacena información sobre las habitaciones disponibles en el hotel.

| Columna         | Tipo de Dato     | Descripción                                      |
|-----------------|------------------|--------------------------------------------------|
| `HabitacionID`  | `INT`            | Identificador único de la habitación (clave primaria). |
| `TipoHabitacion`| `NVARCHAR(50)`   | Tipo de habitación (ej: Individual, Doble, Suite).    |
| `PrecioNoche`   | `DECIMAL(10,2)`  | Precio por noche de la habitación.                   |
| `Estado`        | `NVARCHAR(20)`   | Estado de la habitación (Disponible, Ocupada, Mantenimiento). |

### **2. Tabla `Clientes`**
Almacena información sobre los clientes que realizan reservas.

| Columna         | Tipo de Dato     | Descripción                                      |
|-----------------|------------------|--------------------------------------------------|
| `ClienteID`     | `INT`            | Identificador único del cliente (clave primaria).    |
| `Nombre`        | `NVARCHAR(100)`  | Nombre completo del cliente.                        |
| `Email`         | `NVARCHAR(100)`  | Correo electrónico del cliente.                     |
| `Telefono`      | `NVARCHAR(15)`   | Número de teléfono del cliente.                     |
| `Direccion`     | `NVARCHAR(255)`  | Dirección del cliente.                              |

### **3. Tabla `Reservas`**
Registra las reservas realizadas por los clientes.

| Columna         | Tipo de Dato     | Descripción                                      |
|-----------------|------------------|--------------------------------------------------|
| `ReservaID`     | `INT`            | Identificador único de la reserva (clave primaria). |
| `ClienteID`     | `INT`            | Identificador del cliente (clave foránea).          |
| `HabitacionID`  | `INT`            | Identificador de la habitación (clave foránea).     |
| `FechaEntrada`  | `DATE`           | Fecha de entrada del cliente.                       |
| `FechaSalida`   | `DATE`           | Fecha de salida del cliente.                        |
| `Total`         | `DECIMAL(10,2)`  | Total a pagar por la reserva.                       |

### **4. Tabla `Servicios`**
Almacena los servicios adicionales que ofrece el hotel (ej: desayuno, spa).

| Columna         | Tipo de Dato     | Descripción                                      |
|-----------------|------------------|--------------------------------------------------|
| `ServicioID`    | `INT`            | Identificador único del servicio (clave primaria). |
| `NombreServicio`| `NVARCHAR(100)`  | Nombre del servicio.                              |
| `Precio`        | `DECIMAL(10,2)`  | Precio del servicio.                              |

### **5. Tabla `DetalleServicios`**
Registra los servicios adicionales asociados a una reserva.

| Columna         | Tipo de Dato     | Descripción                                      |
|-----------------|------------------|--------------------------------------------------|
| `DetalleID`     | `INT`            | Identificador único del detalle (clave primaria).  |
| `ReservaID`     | `INT`            | Identificador de la reserva (clave foránea).      |
| `ServicioID`    | `INT`            | Identificador del servicio (clave foránea).       |
| `Cantidad`      | `INT`            | Cantidad del servicio contratado.                 |

---

## **Procedimientos Almacenados**

### **1. `RegistrarReserva`**
Registra una nueva reserva y actualiza el estado de la habitación a "Ocupada".

In [ ]:
CREATE PROCEDURE RegistrarReserva
    @ClienteID INT,
    @HabitacionID INT,
    @FechaEntrada DATE,
    @FechaSalida DATE
AS
BEGIN
    DECLARE @PrecioNoche DECIMAL(10,2);
    DECLARE @DiasEstadia INT;
    DECLARE @Total DECIMAL(10,2);

    -- Obtener el precio por noche de la habitación
    SELECT @PrecioNoche = PrecioNoche
    FROM Habitaciones
    WHERE HabitacionID = @HabitacionID;

    -- Calcular los días de estadía
    SET @DiasEstadia = DATEDIFF(DAY, @FechaEntrada, @FechaSalida);

    -- Calcular el total
    SET @Total = @PrecioNoche * @DiasEstadia;

    -- Insertar la reserva
    INSERT INTO Reservas (ClienteID, HabitacionID, FechaEntrada, FechaSalida, Total)
    VALUES (@ClienteID, @HabitacionID, @FechaEntrada, @FechaSalida, @Total);

    -- Actualizar el estado de la habitación a "Ocupada"
    UPDATE Habitaciones
    SET Estado = 'Ocupada'
    WHERE HabitacionID = @HabitacionID;

    PRINT 'Reserva registrada exitosamente.';
END;
GO

### **2. AgregarServicioAReserva**
Agrega un servicio adicional a una reserva existente y actualiza el total.

In [ ]:
CREATE PROCEDURE AgregarServicioAReserva
    @ReservaID INT,
    @ServicioID INT,
    @Cantidad INT
AS
BEGIN
    DECLARE @PrecioServicio DECIMAL(10,2);

    -- Obtener el precio del servicio
    SELECT @PrecioServicio = Precio
    FROM Servicios
    WHERE ServicioID = @ServicioID;

    -- Insertar el servicio en el detalle de la reserva
    INSERT INTO DetalleServicios (ReservaID, ServicioID, Cantidad)
    VALUES (@ReservaID, @ServicioID, @Cantidad);

    -- Actualizar el total de la reserva
    UPDATE Reservas
    SET Total = Total + (@PrecioServicio * @Cantidad)
    WHERE ReservaID = @ReservaID;

    PRINT 'Servicio agregado a la reserva exitosamente.';
END;
GO

### **3. CancelarReserva**
Cancela una reserva y libera la habitación asociada.

In [ ]:
CREATE PROCEDURE CancelarReserva
    @ReservaID INT
AS
BEGIN
    DECLARE @HabitacionID INT;

    -- Obtener la habitación asociada a la reserva
    SELECT @HabitacionID = HabitacionID
    FROM Reservas
    WHERE ReservaID = @ReservaID;

    -- Eliminar la reserva
    DELETE FROM Reservas
    WHERE ReservaID = @ReservaID;

    -- Actualizar el estado de la habitación a "Disponible"
    UPDATE Habitaciones
    SET Estado = 'Disponible'
    WHERE HabitacionID = @HabitacionID;

    PRINT 'Reserva cancelada y habitación liberada exitosamente.';
END;
GO

## Triggers
### **1. trg_ValidarFechasReserva**
Valida que la fecha de salida sea posterior a la fecha de entrada.

In [ ]:
CREATE TRIGGER trg_ValidarFechasReserva
ON Reservas
FOR INSERT, UPDATE
AS
BEGIN
    IF EXISTS (
        SELECT 1
        FROM inserted
        WHERE FechaSalida <= FechaEntrada
    )
    BEGIN
        RAISERROR('La fecha de salida debe ser posterior a la fecha de entrada.', 16, 1);
        ROLLBACK TRANSACTION;
    END;
END;
GO

### **2. trg_ValidarReservaActiva**
Evita que se agreguen servicios a una reserva cancelada o inexistente.

In [ ]:
CREATE TRIGGER trg_ValidarReservaActiva
ON DetalleServicios
FOR INSERT
AS
BEGIN
    IF EXISTS (
        SELECT 1
        FROM inserted I
        JOIN Reservas R ON I.ReservaID = R.ReservaID
        WHERE R.ReservaID IS NULL
    )
    BEGIN
        RAISERROR('No se pueden agregar servicios a una reserva cancelada o inexistente.', 16, 1);
        ROLLBACK TRANSACTION;
    END;
END;
GO

### **3. trg_ActualizarEstadoHabitacion**
Actualiza el estado de la habitación a "Disponible" cuando se cancela una reserva.

In [ ]:
CREATE TRIGGER trg_ActualizarEstadoHabitacion
ON Reservas
FOR DELETE
AS
BEGIN
    UPDATE Habitaciones
    SET Estado = 'Disponible'
    WHERE HabitacionID IN (SELECT HabitacionID FROM deleted);
END;
GO